In [1]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad

In [16]:
# experimental data
save_folder = 'run7'
n_points = 10000

lower_factor = 0.99
upper_factor = 2 - lower_factor

# Load experimental data
atlas_data = pd.read_csv('../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_15525/4196082941.py:9: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead

/tmp/ipykernel_15525/4196082941.py:10: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead



In [3]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'   

def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=lower_factor, upper_factor=upper_factor):
    # Obtém os parâmetros iniciais
    initial_params = ensemble_parameters[ensemble_name][model_type]
    
    # Cria as variações
    initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
    initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
    return initial_params, initial_params_low, initial_params_high

# Get parameters for selected configuration
initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# Para Atlas
initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)




In [4]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  

def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323


In [5]:
def full_int(mg, a1, a2, m2_func, q_val, sqrt_s):
    # Garante que q_val seja array 1D
    q_val = np.atleast_1d(q_val)
    results = []

    for q in q_val:
        def integrand(y, x, mg, a1, a2, m2_func, q_val):
            k = sqrt_s * x
            phi = 2 * np.pi * y
            jacobian = 2 * np.pi * sqrt_s
            return k * (
                T_1(k, q_val, phi, mg, a1, a2, m2_func)
                - T_2(k, q_val, phi, mg, a1, a2, m2_func)
            ) * jacobian

        def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, m2_func, q),
                0, 1, n=n_points
            )[0]

        integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
        results.append(integral_value)

    # Retorna escalar se apenas um q_val foi passado
    return np.array(results) if len(results) > 1 else results[0]


In [6]:
# def model function
def model_function(x, eps, mg, a1, a2, sqrt_s, model_type='log'):

    # Definindo os parâmetros específicos do modelo
    params = {
        'epsilon': eps,
        'mg': mg,
        'a1': a1,
        'a2': a2
    }
    
    # Escolhendo a massa conforme o modelo
    m2 = m2_log if model_type == 'log' else m2_pl
    
    dif_sigma_lst = []
    
    for q2 in x:
        t = -q2
        
        integral_value = full_int(mg, a1, a2, m2, q2, sqrt_s)

        diff_T = integral_value
        s = sqrt_s ** 2
        amp_value = amp_calculation(diff_T, s, params['epsilon'], t)
        dif_sigma_value = differential_sigma(amp_value, s)
        dif_sigma_lst.append(dif_sigma_value)
    
    return np.array(dif_sigma_lst)

In [7]:
# set cost and minimize
def model_7(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=7000, model_type='pl')

def model_8(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=8000, model_type='pl')

def model_13(x, eps, mg, a1, a2):
    return model_function(x, eps, mg, a1, a2, sqrt_s=13000, model_type='pl')


chi2_7  = LeastSquares(x_7_atlas,  y_7_atlas,  yerr_7_atlas,  model_7)
chi2_8  = LeastSquares(x_8_atlas,  y_8_atlas,  yerr_8_atlas,  model_8)
chi2_13 = LeastSquares(x_13_atlas, y_13_atlas, yerr_13_atlas, model_13)


chi2_total = chi2_7 + chi2_8 + chi2_13


minuit_born = Minuit(
    chi2_total,
    mg = 0.421,
    a1 = 1.517,
    a2 = 2.05,
    eps = 0.0753
)

minuit_born.migrad()
minuit_born.hesse()


┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 25.93 (χ²/ndof = 0.2)      │              Nfcn = 330              │
│ EDM = 0.000111 (Goal: 0.0002)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬──────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼──────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ eps  │  0.0616   │  0.0022   │            │            │         │         │       │
│ 1 │ mg   │   0.389   │   0.005   │            │            │         │         │       │
│ 2 │ a1   │   1.49    │   0.05    │            │            │         │         │       │
│ 3 │ a2   │   2.16    │   0.31    │            │            │         │         │       │
└───┴──────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌─────┬─────────────────────────────────────┐
│     │      eps       mg       a1       a2 │
├─────┼─────────────────────────────────────┤
│ eps │ 4.86e-06    10e-6    54e-6  -158e-6 │
│  mg │    10e-6 2.41e-05 0.056e-3 0.100e-3 │
│  a1 │    54e-6 0.056e-3   0.0023  -0.0136 │
│  a2 │  -158e-6 0.100e-3  -0.0136   0.0954 │
└─────┴─────────────────────────────────────┘

In [8]:
# Calculates and plot dif sigma 
def get_dif_sigma(epsilon, mg, a1, a2, mg_model):

    sqrt_s = 7000
    scale = 1  # caso único
    start_q2 = 0.006
    max_q2   = 0.204
    q2_step  = 0.001

    lst_q2 = []
    lst_dif_sigma = []

    q2 = start_q2
    while q2 <= max_q2:
        t = -q2

        integral_value = full_int(mg, a1, a2, mg_model, q2, sqrt_s)


        diff_T = integral_value
        print(diff_T)
        # print(f"q2: {q2}, diff_T: {diff_T}")

        s          = sqrt_s**2
        amp_value  = amp_calculation(diff_T, s, epsilon, t)
        dif_sigma  = differential_sigma(amp_value, s) * scale

        lst_q2.append(q2)
        lst_dif_sigma.append(dif_sigma)

        q2 += q2_step

    return {sqrt_s: (lst_q2, lst_dif_sigma)}

#for pl atlas
dif_sigma_pl_atlas = get_dif_sigma(
    minuit_born.values['eps'],
    minuit_born.values['mg'], 
    minuit_born.values['a1'],
    minuit_born.values['a2'],
    m2_pl
)

dif_sigma_pl_atlas_7_q2 = dif_sigma_pl_atlas[7000][0]
dif_sigma_pl_atlas_7_values = dif_sigma_pl_atlas[7000][1]


def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))

fig_atlas = go.Figure()


# for pl atlas
add_differential_trace(fig_atlas, dif_sigma_pl_atlas_7_q2, dif_sigma_pl_atlas_7_values,label='7 TeV', color='blue', mg_model='pl')

#data points
add_data_trace(fig_atlas, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

# Atualiza layout
fig_atlas.update_layout(
    title='dσ/dt vs. |t| - Log and PL models in ATLAS',
    xaxis_title='|t| (GeV²)',
    yaxis_title='dσ/dt (mb/GeV²)',
    yaxis_type='log',
    legend_title='Mass Model',
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_atlas.update_xaxes(gridcolor='lightgray')
fig_atlas.update_yaxes(gridcolor='lightgray')

# fig_atlas.show(renderer='browser')


10.020115936124178
9.966872457603387
9.913871227727608
9.861111865947278
9.80859399644589
9.756317247843178
9.704281252887302
9.652485648147463
9.600930073698962
9.549614172814076
9.498537591646093
9.447699978891045
9.397100985501508
9.346740264326542
9.296617469806169
9.246732257660554
9.197084284537773
9.147673207697245
9.098498684692657
9.049560373049777
9.000857929936537
8.952391011842046
8.904159274257676
8.856162371359485
8.808399955692558
8.760871677857303
8.713577186197519
8.666516126490723
8.619688141652647
8.57309287144241
8.526729952161956
8.480599016356962
8.434699692513899
8.38903160476886
8.34359437271512
8.298387611092512
8.253410929449942
8.208663931863247
8.164146216881058
8.119857377151078
8.075796999012335
8.031964662498549
7.98835994115896
7.944982401449771
7.90183160287536
7.858907097829885
7.816208430869928
7.773735139207043
7.731486752114817
7.689462790559945
7.64766276779101
7.606086188118136
7.564732547733985
7.52360133395384
7.4826920251344085
7.44200409105262

In [9]:
# # PLOT BORN SIGMA TOT =============================================================
# # 
# # =============================================================


data_sigma_tot_atlas = pd.read_csv(
    "../../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70
)

x_sigma_tot_atlas = data_sigma_tot_atlas[0].to_numpy()
y_sigma_tot_atlas = data_sigma_tot_atlas[1].to_numpy()
y_error_sigma_tot_atlas = data_sigma_tot_atlas[2].to_numpy()

lst_born_amp = []

start_sqrt_s = 1
max_sqrt_s = 13010
step = 100

def add_total_trace(fig, x, y, color='red', label='', line_style='solid', legend=True, size = 3, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width, dash=line_style),
        marker=dict(size=size),
        name = label, 
        showlegend=legend
    ))

def get_sigma_tot(epsilon, mg, a1, a2, mg_model):

    lst_sigma_tot = []
    lst_sqrt_s = []

    sqrt_s = start_sqrt_s

    while sqrt_s <= max_sqrt_s:

        s = sqrt_s ** 2

        integral_value = full_int(mg, a1, a2, mg_model, 0.0, sqrt_s) 

        born_amp = amp_calculation(integral_value, s, epsilon, 0)
        lst_born_amp.append(born_amp)
        
        lst_sigma_tot.append(sigma_tot(
            amp_calculation(integral_value, s, epsilon, 0), s))
        
        lst_sqrt_s.append(sqrt_s)
        sqrt_s += step
    return lst_sigma_tot, lst_sqrt_s

##-----------------------------------------------------------------------------------------------

sigma_tot_pl_atlas = get_sigma_tot(
    minuit_born.values['eps'],
    minuit_born.values['mg'],
    minuit_born.values['a1'],
    minuit_born.values['a2'],
    m2_pl
)

sigma_tot_pl_atlas_values = sigma_tot_pl_atlas[0]
lst_sqrt_s = sigma_tot_pl_atlas[1]



fig = go.Figure()

add_total_trace(fig, lst_sqrt_s, sigma_tot_pl_atlas_values, color='blue', label='PL Atlas', line_style='solid')

#-----------------------------------------------------------------------------------------------
#----

add_data_trace(fig, x_sigma_tot_atlas, y_sigma_tot_atlas, y_error_sigma_tot_atlas, name='ATLAS', show_label=True, mode='markers')

fig.update_layout(
    title = 'σ_tot vs. √s - Ensemble Atlas and Totem in Log and PL model',
    xaxis=dict(
        title='√s [GeV]',
        type='log',
        range=[np.log10(2000), np.log10(14000)],
    ),
    yaxis=dict(
        title='σ_tot [mb]',
        range=[80, 125]
    ),
    showlegend=True,
    legend=dict(
        title='Ensembles'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
    
fig.update_xaxes(gridcolor='lightgray')
fig.update_yaxes(gridcolor='lightgray')

# fig.show(renderer="browser")

/tmp/ipykernel_15525/414226233.py:6: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead



In [10]:
def full_int(mg, a1, a2, m2_func, q_val, sqrt_s):
    # Garante que q_val seja array 1D
    q_val = np.atleast_1d(q_val)
    results = []

    for q in q_val:
        def integrand(y, x, mg, a1, a2, m2_func, q_val):
            k = sqrt_s * x
            phi = 2 * np.pi * y
            jacobian = 2 * np.pi * sqrt_s
            return k * (
                T_1(k, q_val, phi, mg, a1, a2, m2_func)
                - T_2(k, q_val, phi, mg, a1, a2, m2_func)
            ) * jacobian

        def inner_integral(x):
            return fixed_quad(
                lambda y: integrand(y, x, mg, a1, a2, m2_func, q),
                0, 1, n=n_points
            )[0]

        integral_value = fixed_quad(inner_integral, 0, 1, n=n_points)[0]
        results.append(integral_value)

    # Retorna escalar se apenas um q_val foi passado
    return np.array(results) if len(results) > 1 else results[0]


In [11]:
# from scipy.integrate import quad  # kept for compatibility if used elsewhere

# lst_chi = []
# lst_amp_eik = []
# lst_diff_sigma = []

# lst_q_integration = np.linspace(0, 5, 1000)
# lst_b_integration = np.linspace(0, 15, 1000)

# # step size for Riemann sum
# dq = lst_q_integration[1] - lst_q_integration[0]

# # midpoints for improved Riemann sum accuracy
# q_midpoints = lst_q_integration[:-1] + dq / 2

# for b_val in lst_b_integration:

#     chi_sum = 0

#     for q_val in q_midpoints:

#         q2_val = q_val**2
#         sqrt_s = 7000

#         s = sqrt_s ** 2
#         t = -q2_val

#         diff_t = full_int(
#             minuit_born.values['mg'],
#             minuit_born.values['a1'],
#             minuit_born.values['a2'],
#             m2_pl,
#             q2_val,
#             sqrt_s
#         )

#         born_amp = amp_calculation(diff_t, s, minuit_born.values['eps'], t)

#         chi_val = (1/s) * q_val * j0(b_val * q_val) * born_amp

#         # Midpoint Riemann sum contribution
#         chi_sum += chi_val * dq

#     print(chi_sum)

#     lst_chi.append(chi_sum)


In [12]:
# import numpy as np
# from scipy.integrate import quad
# from scipy.special import j0

# lst_chi = []
# lst_amp_eik = []

# # Integration limits
# q_min, q_max = 0, 0.4
# b_min, b_max = 0, 20

# # Integration grid for outer b loop
# n_points_b = 200
# lst_b = np.linspace(b_min, b_max, n_points_b)

# sqrt_s = 13000
# s = sqrt_s ** 2


# # Define integrand in q for given b
# def chi_integrand_q(q_val, b_val):
#     q2_val = q_val ** 2
#     t = -q2_val

#     diff_t = full_int(
#         minuit_born.values['mg'],
#         minuit_born.values['a1'],
#         minuit_born.values['a2'],
#         m2_pl,
#         q2_val,
#         sqrt_s
#     )

#     born_amp = amp_calculation(diff_t, s, minuit_born.values['eps'], t)
#     return (1 / s) * q_val * j0(b_val * q_val) * born_amp  # complex-valued


# # Outer integration in b (explicit loop)
# def chi_integrand_b(b_val):
#     # Integrate real and imaginary parts separately
#     real_part = lambda q: np.real(chi_integrand_q(q, b_val))
#     imag_part = lambda q: np.imag(chi_integrand_q(q, b_val))

#     chi_real, _ = quad(real_part, q_min, q_max)
#     chi_imag, _ = quad(imag_part, q_min, q_max)

#     return chi_real + 1j * chi_imag


# # Perform full integration in b
# for b_val in lst_b:
#     chi_sum = chi_integrand_b(b_val)
#     print(chi_sum)
#     lst_chi.append(chi_sum)


In [ ]:
from scipy.integrate import fixed_quad
import numpy as np
from scipy.special import j0

b_max = 30
q_min, q_max = 0, 10

lst_chi = []
lst_b_integration = np.linspace(0, b_max, 100)

sqrt_s = 7000
s = sqrt_s ** 2


# Loop over b points (so we can store chi(b) values)
for b_val in lst_b_integration:

    # Inner integration over q using fixed_quad
    def integrand_q(q_val):
    # Força q_val a ser sempre array 1D para compatibilidade
        q_val = np.atleast_1d(q_val)
        results = []
        for q in q_val:
            q2_val = q ** 2
            t = -q2_val

            diff_t = full_int(
                minuit_born.values['mg'],
                minuit_born.values['a1'],
                minuit_born.values['a2'],
                m2_pl,
                q2_val,
                sqrt_s
            )

            born_amp = amp_calculation(diff_t, s, minuit_born.values['eps'], t)

            chi_val = (1/s) * q * j0(b_val * q) * born_amp
            

            results.append(chi_val)

        return np.array(results)

    chi_val, _ = fixed_quad(integrand_q, q_min, q_max, n=1000)

    lst_chi.append(chi_val)

    print(chi_val)


12.511445121296052j
12.482744190525105j
12.397023138474939j
12.25541910095291j
12.059800582617886j
11.812728165421769j
11.517401054255297j
11.17759094918002j
10.797565070021836j
10.3820004336049j
9.93589168800175j
9.464454936114345j
8.973030029878936j
8.466983786606843j
7.951616473565126j
7.432073731728144j
6.913265872976842j
6.399796197185919j
5.895899648454752j
5.40539277604891j
4.931635598722347j
4.477505604187455j
4.045383761172109j
3.637152091236564j
3.2542020513055747j
2.8974527238674446j
2.567377606118926j
2.264038635934118j
1.9871259931455114j
1.7360021688515437j
1.5097488009352855j
1.3072148265742622j
1.1270645966658257j
0.9678247260845667j
0.8279286100578733j
0.7057577128155613j
0.5996789221072558j
0.5080774545246357j
0.42938498470959935j
0.36210285016407895j
0.3048203471704854j
0.25622827805344317j
0.21512803259940078j
0.18043658501054005j
0.15118786154233374j
0.12653098324008075j
0.10572591414426659j
0.08813704993714823j
0.07322526780825242j
0.060538928298693485j
0.04970427

In [ ]:
# import numpy as np
# import matplotlib.pyplot as plt
# from scipy.integrate import fixed_quad
# from scipy.special import j0

# # --- parâmetros já existentes ---
# # q_min, q_max, b_val, s, sqrt_s, m2_pl, minuit_born devem estar definidos
# # integrand_q já definida exatamente como no seu código anterior

# # --- valor de referência altamente preciso (fixed_quad com n=2000) ---
# chi_ref, _ = fixed_quad(integrand_q, q_min, q_max, n=2000)
# chi_ref_imag = np.imag(chi_ref)

# # --- listas de comparação ---
# n_values = [20, 50, 100, 200, 500, 1000]
# errors_riemann = []
# errors_fixed = []

# for n in n_values:
#     # Soma de Riemann
#     q_vals = np.linspace(q_min, q_max, n)
#     dq = (q_max - q_min) / (n - 1)
#     riemann_val = np.sum(integrand_q(q_vals)) * dq
#     err_r = abs((np.imag(riemann_val) - chi_ref_imag) / chi_ref_imag)
#     errors_riemann.append(err_r)

#     # Fixed Quad
#     chi_fixed, _ = fixed_quad(integrand_q, q_min, q_max, n=n)
#     err_f = abs((np.imag(chi_fixed) - chi_ref_imag) / chi_ref_imag)
#     errors_fixed.append(err_f)

# # --- gráfico de erro relativo ---
# plt.figure(figsize=(7, 5))
# plt.loglog(n_values, errors_riemann, 'o--', label='Soma de Riemann')
# plt.loglog(n_values, errors_fixed, 's-', label='Fixed Quad')
# plt.xlabel('Número de pontos de integração (n)')
# plt.ylabel('Erro relativo |Im(χ) - Im(χ_ref)| / |Im(χ_ref)|')
# plt.title('Comparação de convergência: Riemann vs Fixed Quad')
# plt.grid(True, which='both', ls='--', alpha=0.6)
# plt.legend()
# plt.show()


In [34]:
fig_chi = go.Figure()

add_total_trace(fig_chi, lst_b_integration, np.imag(lst_chi), color='red', line_style='solid')

fig_chi.update_layout(
    title = r'$Im(\chi) \quad \text{vs.} \quad b \quad \text{(fixed sqrt(s) = 7000 GeV)}$',
    xaxis=dict(
        title='b'
    ),

    showlegend=True,
    legend=dict(
        title=r'$Im(\chi)$'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
    
fig_chi.update_xaxes(gridcolor='lightgray')
fig_chi.update_yaxes(gridcolor='lightgray')

fig_chi.show(renderer="browser")
# fig_chi.write_image('../../../../results/eikonal/b_values_plots/chi_b.pdf', width=1200, height=600)
# fig_eik.write_html('chi_sum.html')


Opening in existing browser session.
